In [ ]:
# Lab type: extend
# Course: AI402 — Retrieval & RAG Systems
# Lesson: RAG Evals: Faithfulness, Groundedness, and Citations
# Task: Canned claim-decomposition and judge outputs are provided (no API
# key needed). Extend the harness: compute faithfulness, then build the
# citation-support checker the naive version gets wrong.

# Lab: Extending a RAG Eval with Faithfulness and Citation Checks

In production, claim extraction and support verdicts come from versioned LLM judge prompts. Here they are supplied as canned data so the lab runs offline — the *logic* you build around them is exactly the production logic.

**Outputs are cleared.** Run every cell top to bottom.

## Setup: one answer, its context, and the judge's raw output

In [ ]:
!pip install sentence-transformers rank-bm25 faiss-cpu numpy pandas --quiet

In [ ]:
import numpy as np

# The Nimbus Analytics product knowledge base: (doc_id, heading_path, text)
CORPUS = [
    ("plans-overview", "Pricing > Plans",
     "Nimbus Analytics offers three subscription plans: Starter, Teams, and "
     "Enterprise. Starter includes 5 seats and community support. Teams includes "
     "50 seats, shared dashboards, and priority email support. Enterprise includes "
     "unlimited seats, priority support, and advanced security features."),
    ("sso-policy", "Pricing > Enterprise plan",
     "Single sign-on (SSO) with SAML 2.0 is available on the Enterprise plan only. "
     "The Teams plan does not include SSO. Enterprise customers can configure SSO "
     "from the admin console under Security settings."),
    ("seat-pricing", "Pricing > Seats",
     "Per-seat pricing: Starter is $12 per seat per month, Teams is $29 per seat "
     "per month, and Enterprise pricing is custom. Annual billing gives a 20 "
     "percent discount on all plans."),
    ("refund-policy", "Billing > Refunds",
     "Customers can request a full refund within 30 days of purchase. To get your "
     "money back after 30 days, contact billing support; partial refunds are "
     "prorated for annual subscriptions."),
    ("error-e4022", "Troubleshooting > Error codes",
     "Error E4022 means the API rate limit was exceeded. The Starter plan allows "
     "100 requests per minute, Teams 1,000, and Enterprise 10,000. Wait 60 seconds "
     "and retry, or upgrade the plan."),
    ("error-e5001", "Troubleshooting > Error codes",
     "Error E5001 indicates an expired API token. Rotate the token from the admin "
     "console under API settings. Tokens expire after 90 days by default."),
    ("api-export", "API > Export",
     "The export endpoint POST /v2/export creates a CSV export of dashboard data. "
     "Exports are limited to 100,000 rows on Teams and 1 million rows on "
     "Enterprise."),
    ("data-retention", "Security > Data retention",
     "Event data is retained for 13 months on all plans. Enterprise customers can "
     "configure custom retention windows up to 5 years from the admin console."),
    ("priority-support", "Support > Tiers",
     "Priority support with a 4-hour response SLA is included in Teams and "
     "Enterprise plans. Starter includes community support only."),
    ("dashboard-sharing", "Product > Dashboards",
     "Shared dashboards let teammates view and edit the same dashboard. Sharing "
     "outside your workspace requires a public link, available on Teams and "
     "Enterprise."),
    ("audit-logs", "Security > Audit logs",
     "Audit logs record sign-ins, permission changes, and data exports. Audit "
     "logs are an Enterprise-only feature and are retained for 2 years."),
    ("cancel-downgrade", "Billing > Cancellation",
     "You can cancel or downgrade at any time from the billing page. Downgrades "
     "take effect at the end of the current billing period."),
]
DOC_IDS = [d[0] for d in CORPUS]
DOC_TEXTS = [f"{d[1]}: {d[2]}" for d in CORPUS]

# Labelled evaluation queries: (query, set of relevant doc_ids)
EVAL_SET = [
    ("does the teams plan include sso", {"sso-policy"}),
    ("how do I get my money back", {"refund-policy"}),
    ("what does error E4022 mean", {"error-e4022"}),
    ("how long is event data kept", {"data-retention"}),
    ("cost per seat on the teams plan", {"seat-pricing"}),
    ("response time for priority support", {"priority-support"}),
    ("row limit for csv export", {"api-export"}),
    ("rotate an expired api token", {"error-e5001"}),
]
print(f"{len(CORPUS)} documents, {len(EVAL_SET)} labelled queries")

In [ ]:
# The pipeline retrieved these chunks (with stable IDs) ...
RETRIEVED = {
    1: ("sso-policy", "Single sign-on (SSO) with SAML 2.0 is available on "
        "the Enterprise plan only. The Teams plan does not include SSO."),
    2: ("plans-overview", "Teams includes 50 seats, shared dashboards, and "
        "priority email support."),
    3: ("seat-pricing", "Per-seat pricing: Starter is $12 per seat per month, "
        "Teams is $29 per seat per month, and Enterprise pricing is custom."),
}

# ... and generation produced this answer:
ANSWER = (
    "The Teams plan costs $29 per seat per month [3] and includes shared "
    "dashboards [2]. It also includes SSO via SAML 2.0 [1], and all plans "
    "come with a 14-day free trial [2]."
)

# Canned output of the claim-decomposition prompt:
CLAIMS = [
    ("The Teams plan costs $29 per seat per month", [3]),
    ("The Teams plan includes shared dashboards", [2]),
    ("The Teams plan includes SSO via SAML 2.0", [1]),
    ("All plans come with a 14-day free trial", [2]),
]

# Canned per-claim verdicts from the judge prompt (claim checked against
# the retrieved context ONLY — never against world knowledge):
JUDGE_VERDICTS = ["SUPPORTED", "SUPPORTED", "NOT_SUPPORTED", "NOT_SUPPORTED"]
print(f"{len(CLAIMS)} claims, verdicts: {JUDGE_VERDICTS}")

## Extension 1: the faithfulness score

Implement `faithfulness(verdicts)` returning the supported fraction, and print each claim with its verdict. Note which claims failed and why the third one fails even though its sentence *cites* chunk [1].

In [ ]:
def faithfulness(verdicts):
    # TODO: fraction of claims judged SUPPORTED
    pass


<details>
<summary>🔑 Reveal model answer — Extension 1</summary>

```python
def faithfulness(verdicts):
    return verdicts.count("SUPPORTED") / len(verdicts)

for (claim, cites), v in zip(CLAIMS, JUDGE_VERDICTS):
    print(f"  [{v:13}] {claim}")
print(f"faithfulness = {faithfulness(JUDGE_VERDICTS):.2f}")
```

Score: 0.50. Claim 3 contradicts chunk 1 (SSO is Enterprise-only; the answer says Teams has it) — a citation was attached to a claim its source *refutes*. Claim 4 (free trial) appears in no retrieved chunk at all: it may even be true in the world, but faithfulness is measured against the context only.

</details>

## Extension 2: the naive citation checker — and yours

Below is the citation checker the AI assistant originally wrote. Run it, observe that it passes the answer, then extend it: a correct checker verifies each citation points at a chunk that *supports* the citing claim, using the judge verdicts.

In [ ]:
import re

def naive_citation_check(answer, retrieved):
    cited = [int(c) for c in re.findall(r"\[(\d+)\]", answer)]
    return all(c in retrieved for c in cited)

print(f"naive checker passes: {naive_citation_check(ANSWER, RETRIEVED)}")

def citation_support_check(claims, verdicts, retrieved):
    # TODO: return a list of (claim, ok) where ok is True only when the
    # claim's citations exist in `retrieved` AND its verdict is SUPPORTED
    pass


<details>
<summary>🔑 Reveal model answer — Extension 2</summary>

```python
def citation_support_check(claims, verdicts, retrieved):
    results = []
    for (claim, cites), verdict in zip(claims, verdicts):
        ok = all(c in retrieved for c in cites) and verdict == "SUPPORTED"
        results.append((claim, ok))
    return results

for claim, ok in citation_support_check(CLAIMS, JUDGE_VERDICTS, RETRIEVED):
    print(f"  [{'ok' if ok else 'FAIL'}] {claim}")
```

The naive checker validates citation *existence* — every `[n]` points at a real chunk, so it blesses an answer whose key claim is refuted by its own source. Citation presence is not citation correctness.

</details>

## Extension 3: the triad, read together

Suppose this pipeline's eval run reports retrieval recall@5 = 0.95, faithfulness = 0.50 (as computed above), answer relevance = 0.9. In the markdown cell below: name the broken stage, justify it from the score combination, and say what the *dangerous* combination from the lesson would look like instead.

*(Your diagnosis here.)*

<details>
<summary>🔑 Reveal model answer — Extension 3</summary>

Recall is high and relevance is high, but half the claims aren't grounded: **generation is inventing** (and mis-citing) — fix prompting or the model, not retrieval. The dangerous combination is the mirror image: recall *low* with faithfulness and relevance *high* — the model faithfully and fluently answers from the wrong context, every per-answer signal looks green, and only the retrieval metric exposes it.

</details>

## Summary

1. Faithfulness is measured per _______, against the retrieved context only.
2. A citation checker must verify _______, not existence.
3. The triad (retrieval, faithfulness, relevance) is diagnostic because each score isolates a pipeline _______.

<details>
<summary>🔑 Reveal summary answers</summary>

1. **claim** — per-claim verdicts stop one fabrication being averaged away.
2. **support**
3. **stage**

</details>